# EDA Video Temporal

Exploratory analysis for temporal video datasets.

Steps:
- Inventory real/fake video folders.
- Summarize file counts and extensions.
- Inspect available sequence outputs.
- Run the training data audit and summarize outputs.



In [ ]:
from __future__ import annotations

import json
import os
import sys
import subprocess
from pathlib import Path

# Resolve repo root from the notebook location.
REPO_ROOT = Path.cwd()
for parent in [REPO_ROOT] + list(REPO_ROOT.parents):
    if (parent / 'scripts').exists() and (parent / 'notebooks').exists():
        REPO_ROOT = parent
        break

# Ensure local modules are importable.
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / 'src'))

PY = sys.executable

def run(cmd: list[str]) -> None:
    # Run a command from the repo root with PYTHONPATH set.
    env = os.environ.copy()
    env['PYTHONPATH'] = os.pathsep.join([str(REPO_ROOT / 'src'), str(REPO_ROOT)])
    print('$', ' '.join(cmd))
    subprocess.run(cmd, cwd=str(REPO_ROOT), check=True, env=env)

def show_json(rel_path: str) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    try:
        data = json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        print(path.read_text(encoding='utf-8', errors='ignore')[:2000])
        return
    print(json.dumps(data, indent=2))

def list_dir(rel_path: str, limit: int = 20) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    print(f'\n{rel_path}/')
    for item in sorted(path.iterdir())[:limit]:
        print(' -', item.name)


In [ ]:
from collections import Counter
from pathlib import Path

video_root = REPO_ROOT / 'data' / 'raw' / 'vision' / 'video'
sequence_root = REPO_ROOT / 'data' / 'sequences'

summary = {
    'video_root': str(video_root),
    'counts': {},
    'extensions': {},
    'sequences': {},
}

print('Video root:', video_root)
if not video_root.exists():
    print('Missing:', video_root)
else:
    for label in ['real', 'fake']:
        path = video_root / label
        if not path.exists():
            print('Missing:', path)
            continue
        files = [p for p in path.rglob('*') if p.is_file()]
        summary['counts'][label] = len(files)
        ext_counts = Counter(p.suffix.lower() or 'no_ext' for p in files)
        summary['extensions'][label] = dict(ext_counts.most_common(8))
        print(label, 'files:', len(files))
        print('Top extensions:', dict(ext_counts.most_common(5)))

print('')
print('Sequences root:', sequence_root)
if sequence_root.exists():
    seq_files = [p for p in sequence_root.rglob('*') if p.is_file()]
    summary['sequences']['file_count'] = len(seq_files)
    for sample in seq_files[:10]:
        print(' -', sample.relative_to(REPO_ROOT))
else:
    print('Missing:', sequence_root)


In [ ]:
# Sample video file names to verify structure.
if video_root.exists():
    samples = []
    for label in ['real', 'fake']:
        path = video_root / label
        if not path.exists():
            continue
        for file in path.rglob('*'):
            if file.is_file():
                samples.append(str(file.relative_to(REPO_ROOT)))
            if len(samples) >= 12:
                break
        if len(samples) >= 12:
            break
    print('Sample files:')
    for item in samples:
        print(' -', item)


In [ ]:
# Persist summary for quick reference.
report_dir = REPO_ROOT / 'reports'
report_dir.mkdir(parents=True, exist_ok=True)
summary_path = report_dir / 'eda_video_temporal_summary.json'
summary_path.write_text(json.dumps(summary, indent=2))
print('Saved summary to', summary_path)


In [ ]:
# Generate a training data audit
run([PY, 'scripts/training_data_audit.py'])



In [ ]:
# Summarize video-related entries from the training data audit.
audit_path = REPO_ROOT / 'reports' / 'TRAINING_DATA.json'
if not audit_path.exists():
    print('Missing:', audit_path)
else:
    audit = json.loads(audit_path.read_text(encoding='utf-8'))
    items = [
        item for item in audit.get('required', []) + audit.get('optional', [])
        if 'video' in str(item.get('name', '')).lower()
    ]
    if not items:
        print('No video entries found in TRAINING_DATA.json')
    else:
        print('video datasets in audit:')
        for item in items:
            print(' -', item.get('name'), '|', item.get('status'), '|', item.get('path'))


In [ ]:
# Quick artifact index for verification.
for folder in ['models', 'experiments', 'artifacts', 'runs', 'reports', 'logs']:
    path = REPO_ROOT / folder
    if not path.exists():
        continue
    print(f'\n{folder}/')
    for item in sorted(path.iterdir())[:20]:
        print(' -', item.name)
